# Hotel Maintenance Operations Analytics

This notebook analyzes 1,200 synthetic hotel maintenance work orders to find opportunities to reduce downtime, repeat repairs, cost, and guest disruption. The project connects hotel operations with Python and SQL analytics.

**Business question:** How can hotel leadership use maintenance data to improve response performance and the guest experience?

## 1. Import packages and load the data

The `Path` logic lets this notebook run from either the project root or the `notebooks` folder.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "hotel_maintenance_work_orders.csv"

work_orders = pd.read_csv(
    DATA_PATH,
    parse_dates=["opened_at", "closed_at"],
)
print(f"Rows: {len(work_orders):,}")
print(f"Columns: {work_orders.shape[1]}")
work_orders.head()

## 2. Check data quality

Before calculating KPIs, confirm that IDs are unique, timestamps are complete, and key numeric values are reasonable.

In [ ]:
quality_checks = pd.Series({
    "duplicate_work_order_ids": work_orders["work_order_id"].duplicated().sum(),
    "missing_opened_at": work_orders["opened_at"].isna().sum(),
    "missing_system_type": work_orders["system_type"].isna().sum(),
    "negative_costs": (work_orders["total_cost"] < 0).sum(),
    "closed_before_opened": (work_orders["closed_at"] < work_orders["opened_at"]).sum(),
})
quality_checks.to_frame("count")

## 3. Calculate headline KPIs

In [ ]:
kpis = pd.Series({
    "Total work orders": len(work_orders),
    "Total maintenance cost": work_orders["total_cost"].sum(),
    "Average response minutes": work_orders["response_minutes"].mean(),
    "SLA compliance rate": work_orders["sla_met"].mean(),
    "Repeat repair rate": work_orders["repeat_within_30_days"].mean(),
    "Average downtime hours": work_orders["downtime_hours"].mean(),
})
kpis.to_frame("value")

## 4. Which systems drive the most cost?

Grouping by system connects individual work orders to a management-level view.

In [ ]:
system_summary = (
    work_orders.groupby("system_type")
    .agg(
        work_orders=("work_order_id", "count"),
        total_cost=("total_cost", "sum"),
        average_downtime=("downtime_hours", "mean"),
        repeat_rate=("repeat_within_30_days", "mean"),
    )
    .sort_values("total_cost", ascending=False)
)
system_summary.round(2)

In [ ]:
plot_data = system_summary.sort_values("total_cost")
ax = plot_data["total_cost"].plot(kind="barh", color="#2F6690", figsize=(9, 5))
ax.set_title("Total Maintenance Cost by System", loc="left", weight="bold")
ax.set_xlabel("Total cost ($)")
ax.set_ylabel("")
plt.tight_layout();

## 5. Which shift has the greatest response opportunity?

In [ ]:
shift_summary = (
    work_orders.groupby("shift")
    .agg(
        work_orders=("work_order_id", "count"),
        average_response_minutes=("response_minutes", "mean"),
        sla_compliance=("sla_met", "mean"),
        guest_impact_work_orders=("guest_impact", "sum"),
    )
    .reindex(["Day", "Evening", "Overnight"])
)
shift_display = shift_summary.copy()
shift_display["average_response_minutes"] = shift_display["average_response_minutes"].round(1)
shift_display["sla_compliance_pct"] = (shift_display.pop("sla_compliance") * 100).round(1)
shift_display

## 6. How do preventive and corrective work compare?

This is a descriptive comparison. It does not prove that maintenance type caused the cost difference.

In [ ]:
maintenance_summary = (
    work_orders.groupby("maintenance_type")
    .agg(
        work_orders=("work_order_id", "count"),
        average_cost=("total_cost", "mean"),
        average_downtime=("downtime_hours", "mean"),
        repeat_rate=("repeat_within_30_days", "mean"),
    )
)
maintenance_summary.round(2)

## 7. Does asset age relate to repeat repairs?

In [ ]:
age_order = ["1-5 years", "6-10 years", "11-15 years", "16+ years"]
work_orders["asset_age_group"] = pd.cut(
    work_orders["asset_age_years"],
    bins=[0, 5, 10, 15, float("inf")],
    labels=age_order,
)
age_summary = (
    work_orders.groupby("asset_age_group", observed=True)
    .agg(
        work_orders=("work_order_id", "count"),
        average_cost=("total_cost", "mean"),
        repeat_rate=("repeat_within_30_days", "mean"),
    )
)
age_display = age_summary.copy()
age_display["average_cost"] = age_display["average_cost"].round(2)
age_display["repeat_rate_pct"] = (age_display.pop("repeat_rate") * 100).round(1)
age_display

## 8. Recommendations

1. Prioritize HVAC and refrigeration preventive-maintenance reviews for high-cost assets over 10 years old.
2. Test flex coverage or an on-call escalation window for overnight high-priority requests.
3. Require a root-cause review after any repeat repair within 30 days.
4. Track SLA compliance, response time, repeat rate, downtime, and cost by system each month.

### Limitation

The data is synthetic, and this analysis identifies associations rather than causal effects. A real implementation would also include occupancy, staffing, asset IDs, parts availability, and room revenue.